In [ ]:
# Setup. On Colab (Runtime -> Change runtime type -> GPU), this clones the repo and installs the dependencies.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("gpt.py"):
    !git clone https://github.com/yiheinchai/opus-nanochat.git
    %cd opus-nanochat
    !pip install -q tiktoken safetensors pyarrow

# Let's build the post-training of nanochat: from GPT-2 to a ChatGPT

A lecture in the style of Andrej Karpathy's *Neural Networks: Zero to Hero*, following the post-training pipeline of [nanochat](https://github.com/karpathy/nanochat).

## Intro

Hi everyone. So today we are going to continue where Andrej Karpathy's Zero to Hero series left off. If you've been following along, you've built micrograd, you've built makemore, you've built a GPT from scratch on tiny Shakespeare, and in the last video you reproduced GPT-2, the 124 million parameter version. And at the end of all that, you had a base model. A thing that can continue text. But if you actually try to talk to it, you'll find that it's not ChatGPT. It's not an assistant. It just sort of babbles.

So the question for today is: what happens after pretraining? How do you go from a base model, an internet document simulator, to something that you can actually talk to, that answers your questions, that uses tools, and that can even count the number of r's in strawberry? This stage is called post-training, and today we are going to build all of it, from scratch, following Karpathy's nanochat repository.

Now here is my goal for this video, and I want to be very explicit about it. When you're done, you should be able to reproduce every piece of nanochat's post-training yourself, without looking. Not just know the words, but have typed every line, and understand why every line is there. So we're going to do this the same way as in the rest of the series. We'll start from a problem, try the most obvious thing, see it fail, and then fix it. Every piece of code gets written in the notebook first and tested on tiny examples, and only at the end of each chapter will we look at the file in the repository, which will be the exact same code, cleaned up.

Here's what you need. You need the base model from the GPT-2 video, and I've put that model, with two small additions that we'll write together today, in this gpt.py file. It's the same code, the token embeddings, the position embeddings, the twelve transformer blocks, the language model head, with the same weight sharing. And we'll load the original OpenAI weights into it, exactly like we did in that video. Pretraining is the expensive part of the pipeline, it needs a node of eight H100 GPUs for a couple of hours. But post-training is cheap, and everything today runs on a laptop CPU, or on a free Colab GPU if you want it faster.

So here is the roadmap. First, we'll look at what a base model actually does. Then we'll figure out how to turn a conversation into tokens. Then we'll build a mixture of training data, a dataloader for it, and we'll run supervised finetuning. Then we'll write an inference engine with a KV cache and a python tool, evaluations to measure what we did, and finally reinforcement learning, where the model practices and learns from its own successes and failures. Okay, so let's get started.

## The base model is a document simulator

Okay so first, let's load up GPT-2 and sample from it. The Engine here is something we'll write ourselves later in the lecture, for now just think of it as a function that samples tokens from the model. And I'm writing a little complete function that takes a prompt, puts the beginning of sequence token in front of it, and samples greedily, so always the most likely next token.

In [1]:
import torch
from gpt import GPT
from tokenizer import Tokenizer
from engine import Engine

model = GPT.from_pretrained().eval()
tokenizer = Tokenizer()
engine = Engine(model, tokenizer)

def complete(prompt, max_tokens=60):
    ids = [tokenizer.bos] + tokenizer.encode(prompt)
    out, _ = engine.generate_batch(ids, max_tokens=max_tokens, temperature=0.0)
    return tokenizer.decode(out[0][len(ids):])

print(complete("Hello, I'm a language model,", max_tokens=20))

 and I'm a programmer. I'm a big fan of the language model, and I'm a


"Hello, I'm a language model, and I'm a programmer." You'll recognize this from the GPT-2 video, it's exactly the same output, so we know the weights are loaded correctly. Now let's ask it a question.

In [2]:
print(complete("What is the capital of France?"))



The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital


Okay, so actually this is not too bad. It does know that the capital of France is Paris. But then it says it again, and again, and again, forever. And I want you to really think about why. This model was trained on web pages. So when it sees a question, it's asking itself, statistically, what text comes next in a web page that starts like this. It's not answering you. It's simulating a document. And documents on the internet don't end after one sentence, so it has no notion of being done.

Now because it's a document simulator, there's actually a fun trick. We can give it the beginning of a document that looks like a conversation with an assistant, and let it continue that document. This is called few-shot prompting.

In [3]:
prompt = """The following is a conversation between a helpful AI Assistant and a Human.

Human: Hi there! Who are you?
Assistant: Hello! I'm an AI Assistant. How can I help you today?

Human: What is 2+2?
Assistant: 2+2 is 4.

Human: What is the capital of France?
Assistant:"""
print(complete(prompt, max_tokens=40))

 France.

Human: What is the capital of the United States?

Assistant: The United States.

Human: What is the capital of the United Kingdom?

Assistant:


So look at this. It did pick up on the format, it answers in the voice of the Assistant. The answer is "France", which is wrong, it's a small model. But the much bigger problem is what happens next. It writes the next question from the Human, and then the answer of the Assistant, and so on. It's simulating the whole document, both sides of the conversation, because that's all it knows how to do.

So this tells us three things. Number one, the knowledge is already in there, from pretraining. Number two, the model is perfectly happy to play an assistant, if the document looks like a chat. And number three, it has no idea where its turn ends. And that suggests the whole idea of post-training. Instead of tricking the model with a prompt, we just keep training it, on a lot of documents that are conversations, in some very specific format with clearly marked turns. Then conversations become the kind of document it knows.

> **Exercise:** Before we move on, think about how you would do this. If a conversation has to become a single sequence of tokens, what would that sequence look like? And which of those tokens should the model actually be trained on?

## Conversations are just documents

Okay so what is a conversation, as data? In nanochat, and in most datasets on HuggingFace, it's just a dict with a list of messages, and every message has a role, user or assistant, and the content. Our job is to turn that into one sequence of integers, so that we can train on it exactly like we trained on web pages.

So, the obvious first idea. Why not just write it as text? "User: blah, Assistant: blah". That's what our few-shot prompt did. And there are two problems with it. First, the model can't tell a real turn boundary from the text "Assistant:" appearing inside a message. And second, anyone could type "Assistant:" in their message and pretend to be the assistant. So instead, we're going to mark the turns with brand new special tokens, tokens that only we can insert. A user start and a user end around everything the user says, and an assistant start and an assistant end around everything the assistant says. Let's see why they need to be special. Here's what the GPT-2 tokenizer does with the text of our token.

In [4]:
import tiktoken
gpt2 = tiktoken.get_encoding("gpt2")
ids = gpt2.encode_ordinary("<|user_start|>")
print(ids)
print([gpt2.decode([i]) for i in ids])

[27, 91, 7220, 62, 9688, 91, 29]
['<', '|', 'user', '_', 'start', '|', '>']


Seven ordinary tokens. A less than sign, a pipe, the word user, and so on. That's no good. We want each of our markers to be exactly one token, one integer that can never come out of normal text. And tiktoken lets us do that, we just register them as special tokens.

In [5]:
SPECIAL_TOKENS = ["<|user_start|>", "<|user_end|>", "<|assistant_start|>", "<|assistant_end|>",
                  "<|python_start|>", "<|python_end|>", "<|output_start|>", "<|output_end|>"]
special = {"<|endoftext|>": 50256}
special.update({name: 50257 + i for i, name in enumerate(SPECIAL_TOKENS)})
enc = tiktoken.Encoding(name="gpt2_chat", pat_str=gpt2._pat_str,
                        mergeable_ranks=gpt2._mergeable_ranks, special_tokens=special)
print(special)
print(enc.encode_single_token("<|assistant_end|>"))
print(enc.encode_ordinary("<|assistant_end|>"))

{'<|endoftext|>': 50256, '<|user_start|>': 50257, '<|user_end|>': 50258, '<|assistant_start|>': 50259, '<|assistant_end|>': 50260, '<|python_start|>': 50261, '<|python_end|>': 50262, '<|output_start|>': 50263, '<|output_end|>': 50264}
50260
[27, 91, 562, 10167, 62, 437, 91, 29]


So GPT-2 already has one special token, end of text, with id 50,256, and we'll use it as our beginning of sequence token, BOS. And we give our eight new tokens the next eight ids, 50,257 through 50,264. You'll also notice there are two more pairs, python start and end, and output start and end. Those are for tool use, and we'll get to them in a bit. Now if we ask for the special token, we get a single id. But encode_ordinary, which is what we'll use for everything a user types, still treats it as plain text. So a user can't inject a turn boundary. That's exactly what we want.

Okay, now let's write the function that renders a conversation into tokens. Simplest version first: BOS, and then for every message, the start token, the tokens of the content, and the end token.

In [6]:
def render_conversation(conversation):
    s = enc.encode_single_token
    ids = [50256] # every conversation starts with the BOS token
    for message in conversation["messages"]:
        if message["role"] == "user":
            ids += [s("<|user_start|>"), *enc.encode_ordinary(message["content"]), s("<|user_end|>")]
        else:
            ids += [s("<|assistant_start|>"), *enc.encode_ordinary(message["content"]), s("<|assistant_end|>")]
    return ids

conversation = {"messages": [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris"},
]}
ids = render_conversation(conversation)
print(ids)
print([enc.decode([i]) for i in ids])

[50256, 50257, 2061, 318, 262, 3139, 286, 4881, 30, 50258, 50259, 40313, 50260]
['<|endoftext|>', '<|user_start|>', 'What', ' is', ' the', ' capital', ' of', ' France', '?', '<|user_end|>', '<|assistant_start|>', 'Paris', '<|assistant_end|>']


So the whole conversation is thirteen tokens. And the decoded pieces look right: BOS, user start, the question, user end, assistant start, Paris, assistant end.

Now here's the important question, the one from the exercise. If we train on this sequence like we train on a web page, the model learns to predict every one of these tokens, including the user's question. Is that what we want? Think about it. We don't want a model that simulates users, we want a model that responds to them. Predicting what people ask is not useless, but it's not our job, and with a small model, every bit of capacity matters. So we only want to train on the tokens that the assistant writes. And the way we do that is we return a second list, a mask, with a one for every token the assistant should learn to produce, and a zero for everything else.

In [7]:
def render_conversation(conversation):
    s = enc.encode_single_token
    ids, mask = [], []
    def add_tokens(token_ids, mask_val):
        ids.extend(token_ids)
        mask.extend([mask_val] * len(token_ids))
    add_tokens([50256], 0)
    for message in conversation["messages"]:
        if message["role"] == "user":
            add_tokens([s("<|user_start|>"), *enc.encode_ordinary(message["content"]), s("<|user_end|>")], 0)
        else:
            add_tokens([s("<|assistant_start|>")], 0)
            add_tokens(enc.encode_ordinary(message["content"]), 1)
            add_tokens([s("<|assistant_end|>")], 1)
    return ids, mask

ids, mask = render_conversation(conversation)
for i, m in zip(ids, mask):
    print(f"{enc.decode([i])!r:24} mask={m}")

'<|endoftext|>'          mask=0
'<|user_start|>'         mask=0
'What'                   mask=0
' is'                    mask=0
' the'                   mask=0
' capital'               mask=0
' of'                    mask=0
' France'                mask=0
'?'                      mask=0
'<|user_end|>'           mask=0
'<|assistant_start|>'    mask=0
'Paris'                  mask=1
'<|assistant_end|>'      mask=1


So everything the user wrote, and all the special tokens around it, get a zero. The assistant start token also gets a zero, because we always put it there ourselves, the model never has to produce it. Then the content gets a one, and then assistant end also gets a one. And this one is really important. This is how the model learns to stop talking. Remember the base model that went on forever? This is the fix. After Paris, the model must learn to emit assistant end, and then we stop sampling, and it's the user's turn.

Now for tool use. Sometimes we want the assistant to run a bit of python, for example a calculator. So the content of an assistant message can also be a list of parts. A text part is just text. A python part is code that the assistant writes, and we wrap it in python start and python end. And a python output part is the result of running that code, wrapped in output start and output end. Now pause and think about the mask. Which of these should get a one?

In [8]:
def render_conversation(conversation):
    s = enc.encode_single_token
    ids, mask = [], []
    def add_tokens(token_ids, mask_val):
        ids.extend(token_ids)
        mask.extend([mask_val] * len(token_ids))
    add_tokens([50256], 0)
    for message in conversation["messages"]:
        content = message["content"]
        if message["role"] == "user":
            add_tokens([s("<|user_start|>"), *enc.encode_ordinary(content), s("<|user_end|>")], 0)
            continue
        add_tokens([s("<|assistant_start|>")], 0)
        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        for part in content:
            value = enc.encode_ordinary(part["text"])
            if part["type"] == "text":
                add_tokens(value, 1)
            elif part["type"] == "python": # the assistant writes the code: train on it
                add_tokens([s("<|python_start|>"), *value, s("<|python_end|>")], 1)
            elif part["type"] == "python_output": # python writes the output: don't train on it
                add_tokens([s("<|output_start|>"), *value, s("<|output_end|>")], 0)
        add_tokens([s("<|assistant_end|>")], 1)
    return ids, mask

def visualize(ids, mask):
    GREEN, RED, RESET = '\033[92m', '\033[91m', '\033[0m'
    return '|'.join(f"{GREEN if m else RED}{enc.decode([i])}{RESET}" for i, m in zip(ids, mask))

tool_conversation = {"messages": [
    {"role": "user", "content": "What is 123 times 45?"},
    {"role": "assistant", "content": [
        {"type": "text", "text": "Let me use python: "},
        {"type": "python", "text": "123*45"},
        {"type": "python_output", "text": "5535"},
        {"type": "text", "text": " The answer is 5535."},
    ]},
]}
print(visualize(*render_conversation(tool_conversation)))

<|endoftext|>|<|user_start|>|What| is| 123| times| 45|?|<|user_end|>|<|assistant_start|>|Let| me| use| python|:| |<|python_start|>|123|*|45|<|python_end|>|<|output_start|>|55|35|<|output_end|>| The| answer| is| 55|35|.|<|assistant_end|>


The code gets a one, because at test time the model has to write it. But the output gets a zero. At test time, python is going to produce those tokens, not the model. If we trained on them, we would be teaching the model to predict what python would say, in other words to hallucinate tool outputs, and that's actively harmful. And then after the output, the model continues in green, and it can copy the result into its answer.

And one last function. At test time, we have a conversation where it's the assistant's turn. So we render everything except the last assistant message, and then we append assistant start. And that's our prompt. The model takes it from there, and we sample until it emits assistant end.

In [9]:
def render_for_completion(conversation):
    ids, _ = render_conversation({"messages": conversation["messages"][:-1]})
    return ids + [enc.encode_single_token("<|assistant_start|>")]

print([enc.decode([i]) for i in render_for_completion(conversation)])

['<|endoftext|>', '<|user_start|>', 'What', ' is', ' the', ' capital', ' of', ' France', '?', '<|user_end|>', '<|assistant_start|>']


So the prompt ends with assistant start, and the next token the model writes is the beginning of its answer. Now, I've put exactly this code into the tokenizer file of the repo, and let's just check that our notebook version and the repo version agree.

In [10]:
print(tokenizer.render_conversation(tool_conversation) == render_conversation(tool_conversation))
print(tokenizer.render_for_completion(conversation) == render_for_completion(conversation))

True
True


Good. The repo version, in tokenizer.py, is the same function, with two small additions: if a conversation starts with a system message, we just glue it onto the first user message, because we don't have special tokens for system messages. And there's a max tokens argument that truncates, which, I'll tell you right now, is going to bite us later. And by the way, this exact format is a design choice. OpenAI's ChatML uses tokens like im start and im end, Llama has its own, and they all do basically the same thing. What matters is that you render conversations the same way at training time and at test time. Even a single space different can quietly hurt your model.

> **Exercise:** Add support for system messages to your render_conversation, the way the repo does it. Then write a conversation with two user turns and two assistant turns and visualize it. Which tokens are green?

## Brain surgery on GPT-2

Okay so we have our token ids, but now we have a problem. GPT-2's vocabulary has 50,257 tokens, so its embedding table has exactly 50,257 rows, and our special tokens have ids 50,257 through 50,264. There's no row for them. If we feed id 50,257 into the original GPT-2, it's an index out of range error.

Now if you remember the GPT-2 reproduction video, we padded the vocab size from 50,257 up to 50,304, because it's a nice number, divisible by 128, and the GPU kernels like that. And we said those extra 47 rows are never used. Well, today they get used. So when we load the OpenAI weights, we copy their 50,257 rows, and we have to decide what to put in the new rows. Here's the code. Copy GPT-2's rows, and then, for all the new rows, I'm using the average of all the token embeddings. Why the average? Because random vectors could be anywhere, but the average sort of lives in the region where real embeddings live, so it won't blow anything up. Let's check that it did what we wanted.

In [11]:
w = model.transformer.wte.weight
print(w.shape)
print(model.lm_head.weight is w)
print((w[50257:] - w[:50257].mean(dim=0)).abs().max().item())

torch.Size([50304, 768])
True
0.0


So the table has 50,304 rows. The language model head is literally the same tensor as the embedding table, that's the weight sharing from the GPT-2 paper. And that matters a lot here, because it means the new rows are not only the input embeddings of our special tokens, they're also the output weights that produce their logits. And all the new rows are exactly the average.

Now, and this is something that Karpathy drills into you in the makemore videos, before we train anything, let's sanity check the loss at initialization. What do we expect? If the model had no idea and spread its probability evenly over the vocabulary, the loss would be minus log of one over 50,304, so about 10.8. Let's look at the loss of every single token of our conversation.

In [12]:
import math
print(-math.log(1 / 50304))
ids, mask = render_conversation(conversation)
x, y = torch.tensor([ids[:-1]]), torch.tensor([ids[1:]])
with torch.no_grad():
    losses = model(x, y, loss_reduction='none')
for token, m, loss in zip(ids[1:], mask[1:], losses):
    print(f"{enc.decode([token])!r:24} mask={m} loss={loss.item():.2f}")

10.825839875788878
'<|user_start|>'         mask=0 loss=14.85
'What'                   mask=0 loss=9.90
' is'                    mask=0 loss=1.81
' the'                   mask=0 loss=1.79
' capital'               mask=0 loss=8.83
' of'                    mask=0 loss=1.02
' France'                mask=0 loss=3.61
'?'                      mask=0 loss=0.49
'<|user_end|>'           mask=0 loss=15.17
'<|assistant_start|>'    mask=0 loss=15.26
'Paris'                  mask=1 loss=6.44
'<|assistant_end|>'      mask=1 loss=14.85


So 10.8 is the uniform guess. The normal English tokens look fine, the model is quite sure that "is" follows "What", and so on. But look at the special tokens. All of them are around 15, which is much worse than a uniform guess. The model is confidently wrong about them. And it's because of the weight sharing. The logit of a special token is the dot product of the hidden state with that average embedding, and that just happens to be a small number compared to the logits of real tokens. So the probability is tiny, and the loss is huge. And what this means is that at the start of finetuning, a lot of the loss is going to come from these tokens, and the first thing the model will do is learn that they exist. It's the hockey stick from makemore, all over again.

> **Exercise:** Can you find a better initialization for the new rows, one that gives a loss of about 10.8 or lower for the special tokens at initialization? Try initializing assistant end to the embedding of a newline, or scaling the average vector. Measure the loss with the cell above.

## The data mixture

Okay, now we need conversations to train on. A lot of them. And here's the key idea of the SFT stage in nanochat: the data is a mixture of several datasets, and every one of them teaches the model one specific thing. So let's build them one at a time. And the one I find the most instructive is grade school math, GSM8K, so let's start there.

All of these datasets live on HuggingFace, and every dataset on HuggingFace has an automatic export to parquet files. So I wrote a tiny function that downloads those parquet files and reads them with pyarrow. You don't need the datasets library for this. Let's load the GSM8K training set and look at one problem.

In [13]:
from tasks import load_hub_dataset
table = load_hub_dataset("openai/gsm8k", "main", split="train")
print(table.num_rows, table.column_names)
row = {col: table[col][3].as_py() for col in table.column_names}
print(row["question"])
print(row["answer"])

7473 ['question', 'answer']
Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
Maila read 12 x 2 = <<12*2=24>>24 pages today.
So she was able to read a total of 12 + 24 = <<12+24=36>>36 pages since yesterday.
There are 120 - 36 = <<120-36=84>>84 pages left to be read.
Since she wants to read half of the remaining pages tomorrow, then she should read 84/2 = <<84/2=42>>42 pages.
#### 42


7,473 problems, each with a question and an answer. And look at the answer. What are these things in double angle brackets? These are calculator annotations, from the people who made the dataset. "12 times 2 equals 24" is written out as the expression and its result. And at the very end, after four hash signs, is the final answer, which is what we'll use for grading.

So here's the plan. These calculator annotations are a gift. They're exactly the tool calls that we want our model to learn. So we're going to turn every one of them into a python part, followed by a python output part. Step one, let's split the answer into the text and the annotations. A regex split with a capture group keeps the separators, so we get both.

In [14]:
import re
chunks = re.split(r'(<<[^>]+>>)', row["answer"])
print(chunks)

['Maila read 12 x 2 = ', '<<12*2=24>>', '24 pages today.\nSo she was able to read a total of 12 + 24 = ', '<<12+24=36>>', '36 pages since yesterday.\nThere are 120 - 36 = ', '<<120-36=84>>', '84 pages left to be read.\nSince she wants to read half of the remaining pages tomorrow, then she should read 84/2 = ', '<<84/2=42>>', '42 pages.\n#### 42']


So we get the plain text, then the annotation, then text again, alternating. Step two, for every annotation, we strip the angle brackets and split at the last equals sign. Left of it is the expression, which becomes a python part, and right of it is the result, which becomes a python output part. Everything else is a text part. And then we can render it with our own render_conversation and look at the mask.

In [15]:
parts = []
for chunk in chunks:
    if chunk.startswith('<<') and chunk.endswith('>>'):
        expr, _, result = chunk[2:-2].rpartition('=')
        parts.append({"type": "python", "text": expr})
        parts.append({"type": "python_output", "text": result})
    else:
        parts.append({"type": "text", "text": chunk})
math_conversation = {"messages": [{"role": "user", "content": row["question"]},
                                  {"role": "assistant", "content": parts}]}
print(visualize(*render_conversation(math_conversation)))

<|endoftext|>|<|user_start|>|Jul|ie| is| reading| a| 120|-|page| book|.| Yesterday|,| she| was| able| to| read| 12| pages| and| today|,| she| read| twice| as| many| pages| as| yesterday|.| If| she| wants| to| read| half| of| the| remaining| pages| tomorrow|,| how| many| pages| should| she| read|?|<|user_end|>|<|assistant_start|>|Mail|a| read| 12| x| 2| =| |<|python_start|>|12|*|2|<|python_end|>|<|output_start|>|24|<|output_end|>|24| pages| today|.|
|So| she| was| able| to| read| a| total| of| 12| +| 24| =| |<|python_start|>|12|+|24|<|python_end|>|<|output_start|>|36|<|output_end|>|36| pages| since| yesterday|.|
|There| are| 120| -| 36| =| |<|python_start|>|120|-|36|<|python_end|>|<|output_start|>|84|<|output_end|>|84| pages| left| to| be| read|.|
|Since| she| wants| to| read| half| of| the| remaining| pages| tomorrow|,| then| she| should| read| 84|/|2| =| |<|python_start|>|84|/|2|<|python_end|>|<|output_start|>|42|<|output_end|>|42| pages|.|
|####| 42|<|assistant_end|>


So the model is trained to write the expression, in green, the result is red, it will come from python, and then the model copies the result into its sentence. This is the simplest possible version of tool use. The model learns to hand the arithmetic, which it's bad at, to a calculator, which is perfect at it. And at the end, the four hash signs and the answer. Now to grade an answer, we just pull out the number after the hash signs, with a regex, and compare it to the reference.

In [16]:
ANSWER_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
def extract_answer(completion):
    match = ANSWER_RE.search(completion)
    return match.group(1).strip().replace(",", "") if match else None

print(extract_answer(row["answer"]), extract_answer("so the answer is #### 1,024"), extract_answer("no idea"))

42 1024 None


So 42 for our problem, commas get removed so 1,024 becomes 1024, and None when there's no answer at all. That's it, that's our grader. And notice that this grader is going to be our reward in reinforcement learning, at the end of the lecture.

The second dataset is MMLU, which is multiple choice questions. And it teaches the model the format of multiple choice, which is how we'll evaluate it later. Let's write the function that renders a multiple choice question, and I'll show you two little details in it.

In [17]:
def render_mc(question, letters, choices):
    query = f"Multiple Choice question: {question}\n"
    query += "".join(f"- {choice}={letter}\n" for letter, choice in zip(letters, choices))
    query += "\nRespond only with the letter of the correct answer."
    return query

print(render_mc("What is the capital of France?", "ABCD", ["Berlin", "Madrid", "Paris", "Rome"]))
print(enc.encode_ordinary(" A"), enc.encode_ordinary("A"), enc.encode_ordinary("=A"))

Multiple Choice question: What is the capital of France?
- Berlin=A
- Madrid=B
- Paris=C
- Rome=D

Respond only with the letter of the correct answer.
[317] [32] [28, 32]


The first detail is that the letter comes after the choice, not before it. Big models don't care, but for small models it helps to have the letter right next to the thing it labels. The second detail is that there's no space between the equals sign and the letter. Why? Look at the tokens. "Space A" is token 317, and "A" without a space is token 32. These are completely different tokens as far as the model is concerned, just different integers. The assistant answers with just "A", token 32. So in the prompt, we want the exact same token 32 to appear right next to the choice, so the model can sort of point at it. If you remember the tokenization video, so much weirdness in LLMs comes from the tokenizer, and this is one of those details.

The third dataset is SmolTalk, from HuggingFace. That's 460,000 general conversations, and it's where most of the personality of the assistant comes from.

In [18]:
from tasks import SmolTalk
smoltalk = SmolTalk("train")
for message in smoltalk[0]["messages"]:
    print(f"--- {message['role']}:\n{message['content'][:400]}")

--- user:
A researcher interested in examining students' study time per day collected data from a random sample of 20 students. Here are the data: 

3, 1, 2, 4, 5, 2, 3, 4, 2, 1, 2, 3, 4, 6, 3, 2, 2, 1, 5, 4

What is the mean of this dataset? How would you describe or interpret the result?
--- assistant:
To calculate the mean, we need to add up all the values and divide by the total number of observations. 

The sum of the values is 3 + 1 + 2 + 4 + 5 + 2 + 3 + 4 + 2 + 1 + 2 + 3 + 4 + 6 + 3 + 2 + 2 + 1 + 5 + 4 = 59.

There are 20 observations in the dataset. The mean is calculated by dividing the sum by the number of observations: 59 / 20 = 2.95.

The mean of 2.95 hours per day indicates that, on a
--- user:
How would you describe the dispersion of this data set. Which measurement or calculation of dispersion would you use to describe and or interpret the dispersion of the data.
--- assistant:
To describe the dispersion of the data, I would calculate the standard deviation, as it is a m

These were mostly generated by bigger language models, and then filtered. And it's worth pausing on this, because Karpathy makes a really important point about it in his talks. When you talk to ChatGPT, who are you actually talking to? You're talking to a statistical simulation of this kind of data. Originally it was written by human labelers following long instructions from a company about being helpful and harmless, and nowadays a lot of it is written by other LLMs. But either way, the knowledge comes from pretraining, and the personality, the tone, the length of the answers, all comes from imitating this data.

And the fourth dataset is my favorite. How many r's are there in strawberry? Famously, LLMs get this wrong, and now we can understand why. Look at how it's tokenized.

In [19]:
print([enc.decode([t]) for t in enc.encode_ordinary(" strawberry")])
print([enc.decode([t]) for t in enc.encode_ordinary("strawberry")])

[' strawberry']
['st', 'raw', 'berry']


With a space in front, strawberry is a single token. One integer. The model never sees the letters! To count the r's, it would have to have memorized the spelling of every one of its fifty thousand tokens, and then count in its head, in a single forward pass. And without a space, it's three tokens, which is a completely different set of integers again.

So nanochat has a synthetic task for this, called SpellingBee. Synthetic means we generate the data ourselves, as much as we want. And the interesting part is designing the ideal answer. We know the model can't do this in one step, so we write out a procedure that it can do: spell the word out, then go through the letters one at a time and keep a running count, then double check with python, and only then give the answer. Let's write it.

In [20]:
def spelling_answer(word, letter):
    text = f"We are asked to find the number '{letter}' in the word '{word}'. Let me try a manual approach first.\n\n"
    text += f"First spell the word out:\n{word}:{','.join(word)}\n\nThen count the occurrences of '{letter}':\n"
    count = 0
    for i, char in enumerate(word, 1):
        if char == letter:
            count += 1
            text += f"{i}:{char} hit! count={count}\n"
        else:
            text += f"{i}:{char}\n"
    text += f"\nThis gives us {count}.\n\nLet me double check this using Python:\n\n"
    return [
        {"type": "text", "text": text},
        {"type": "python", "text": f"'{word}'.count('{letter}')"},
        {"type": "python_output", "text": str(word.count(letter))},
        {"type": "text", "text": f"\n\nPython gives us {word.count(letter)}.\n\nMy final answer is:\n\n#### {word.count(letter)}"},
    ]

for part in spelling_answer("strawberry", "r"):
    print(f"[{part['type']}] {part['text']}")

[text] We are asked to find the number 'r' in the word 'strawberry'. Let me try a manual approach first.

First spell the word out:
strawberry:s,t,r,a,w,b,e,r,r,y

Then count the occurrences of 'r':
1:s
2:t
3:r hit! count=1
4:a
5:w
6:b
7:e
8:r hit! count=2
9:r hit! count=3
10:y

This gives us 3.

Let me double check this using Python:


[python] 'strawberry'.count('r')
[python_output] 3
[text] 

Python gives us 3.

My final answer is:

#### 3


First, the spelling. Then the counting, one letter per line, with a running count, so that no single token ever needs much computation. Then python checks it, and the answer goes after the hash signs, so the same grader as GSM8K works. And I think this is a really nice example of how you have to think about the cognition of the model. Every token only gets a fixed amount of compute, one forward pass through twelve layers. So you want to spread the computation out over many tokens. If you ask the model to blurt out the answer in the first token, you're asking it to do all of the work in one forward pass, and it just can't. In the repo, SpellingBee is this same function, plus a random word out of a list of 370,000 English words, a random letter, and a bunch of different ways of asking the question.

Okay, and the last piece is to mix all of these together. A mixture is just a big list of every example in every task, shuffled with a fixed seed. And if you want more of something, you just put it in twice.

In [21]:
import random
from tasks import MMLU, GSM8K, SpellingBee

class TaskMixture:
    def __init__(self, tasks):
        self.tasks = tasks
        self.index_map = [(t, i) for t, task in enumerate(tasks) for i in range(len(task))]
        random.Random(42).shuffle(self.index_map) # mix the tasks together, the same way every time
    def __len__(self):
        return len(self.index_map)
    def __getitem__(self, index):
        t, i = self.index_map[index]
        return self.tasks[t][i]

mixture = TaskMixture([SmolTalk("train", stop=20000), MMLU("auxiliary_train", stop=20000),
                       GSM8K("train"), GSM8K("train"), SpellingBee(size=20000, split="train")])
print(len(mixture))
print([type(mixture.tasks[t]).__name__ for t, _ in mixture.index_map[:8]])

74946
['SpellingBee', 'SpellingBee', 'GSM8K', 'SmolTalk', 'GSM8K', 'SmolTalk', 'MMLU', 'MMLU']


So about 75,000 conversations, and the tasks are nicely interleaved. Now, I'm using small slices of SmolTalk and MMLU here, because we'll train on a CPU. In nanochat, on a GPU, it's all 460,000 SmolTalk conversations, and here's the mixture in our chat_sft script. And remember what this mixture looks like, because it's going to matter later. Twenty thousand SpellingBee examples is more than a quarter of everything.

> **Exercise:** Write your own synthetic task. For example, a task that teaches the model its own name and who made it, or one that reverses a word letter by letter. Generate the ideal answer with code, and think about how to spread the work across tokens.

## The SFT dataloader

Okay, so now we need to turn conversations into batches. And in the pretraining videos this was trivial. We had one endless stream of text, and we just chopped it into rows of 1024 tokens. A document cut in half, who cares. But a conversation is different. If you cut it in half, you get a question without its answer, and an answer without its question. So we want every conversation in one piece, and we want every row to start with a BOS token, because that's what the model will see at test time.

So the first question is, how long are conversations? Let's measure.

In [22]:
lengths = [len(tokenizer.render_conversation(mixture[i], max_tokens=10**6)[0]) for i in range(2000)]
lengths.sort()
print(f"median {lengths[1000]}, 90th percentile {lengths[1800]}, max {lengths[-1]}")
print(f"fraction longer than 512 tokens: {sum(l > 512 for l in lengths) / len(lengths):.3f}")

median 217, 90th percentile 1072, max 20317
fraction longer than 512 tokens: 0.202


Half of them are shorter than about 200 tokens. But there's a long tail, ten percent are over a thousand tokens, and the longest is 20,000. And with the rows of 512 tokens we're going to use on the CPU, one in five conversations won't fit. Keep that number in mind.

Okay so the simplest thing we could do is one conversation per row, and fill up the rest of the row with padding. And we don't train on the padding, so it gets a mask of zero. Let's measure how much of the batch is wasted.

In [23]:
def naive_batch(dataset, start, batch_size, seq_len):
    rows, masks = [], []
    for i in range(start, start + batch_size):
        ids, mask = tokenizer.render_conversation(dataset[i], max_tokens=seq_len + 1)
        pad = seq_len + 1 - len(ids)
        rows.append(ids + [tokenizer.bos] * pad)
        masks.append(mask + [0] * pad)
    return torch.tensor(rows), torch.tensor(masks)

rows, masks = naive_batch(mixture, 0, 64, 512)
print(f"padding: {(rows == tokenizer.bos).float().mean():.3f}, trained on: {masks.float().mean():.3f}")

padding: 0.473, trained on: 0.255


47 percent of every batch is padding. Half of our compute would be spent on nothing. And only a quarter of the tokens are actually trained on. That's terrible.

So the better idea is packing. We put several whole conversations into every row. And the algorithm nanochat uses is called best fit. We keep a buffer of a hundred conversations, and to fill a row, we keep grabbing the longest conversation in the buffer that still fits in the space that's left. And only when nothing fits anymore, we pad the rest of the row. Long conversations go first, and the short ones fill in the gaps. And we never cut a conversation.

But before I show you the code, I want to tell you about a bug I hit writing this, because it's really instructive. In my first version, I rendered every conversation with max tokens set to the row length, so long conversations just got truncated. Seems harmless, right? Let's see what that does to a long multiple choice question.

In [24]:
long_mcq = next(c for c in MMLU("auxiliary_train") if len(tokenizer.render_conversation(c, max_tokens=10**6)[0]) > 600)
ids, mask = tokenizer.render_conversation(long_mcq, max_tokens=513)
print(len(ids), sum(mask))
import torch.nn.functional as F
logits = torch.randn(1, 4, 50304)
print(F.cross_entropy(logits.view(-1, 50304), torch.tensor([-1, -1, -1, -1]), ignore_index=-1))

513 0
tensor(nan)


513 tokens, and a mask that sums to zero. Not a single trainable token! These MMLU training questions include long reading passages, and the answer, the one green token, is at the very end. So truncation cut off the answer, and the whole conversation became prompt. And here's the evil part. The best fit algorithm loves these, because they're exactly the length of a row. So it filled whole batches with them, and the loss on a batch with zero trainable tokens is zero divided by zero, which is not a number. My loss was NaN, and it took me a minute to figure out why. So the fix is simple: if a conversation doesn't fit in a row, we skip it. Okay, now let's write the real thing.

In [25]:
def sft_data_generator(dataset, batch_size, seq_len, buffer_size=100):
    row_capacity = seq_len + 1 # +1 because the targets are the inputs shifted by one
    buffer, cursor = [], 0
    while True:
        rows, mask_rows = [], []
        for _ in range(batch_size):
            row, mask_row = [], []
            while True:
                while len(buffer) < buffer_size:
                    ids, mask = tokenizer.render_conversation(dataset[cursor % len(dataset)], max_tokens=row_capacity + 1)
                    if len(ids) <= row_capacity: # skip the (rare) conversations too long to ever fit
                        buffer.append((ids, mask))
                    cursor += 1
                remaining = row_capacity - len(row)
                fits = [i for i, (ids, _) in enumerate(buffer) if len(ids) <= remaining]
                if not fits:
                    break
                ids, mask = buffer.pop(max(fits, key=lambda i: len(buffer[i][0])))
                row += ids
                mask_row += mask
            padding = row_capacity - len(row)
            rows.append(row + [tokenizer.bos] * padding)
            mask_rows.append(mask_row + [0] * padding) # never train on the padding
        batch, mask = torch.tensor(rows), torch.tensor(mask_rows)
        inputs, targets = batch[:, :-1], batch[:, 1:].clone()
        targets[mask[:, 1:] == 0] = -1 # only train on the assistant's tokens: -1 is ignored by the loss
        yield inputs, targets, cursor / len(dataset)

loader = sft_data_generator(mixture, batch_size=64, seq_len=512)
x, y, _ = next(loader)
x, y, _ = next(loader)
print(x.shape, y.shape)
print(f"padding: roughly {(x == tokenizer.bos).float().mean():.3f}, trained on: {(y >= 0).float().mean():.3f}")

torch.Size([64, 512]) torch.Size([64, 512])
padding: roughly 0.081, trained on: 0.419


So let's read it. The rows have one extra token, because the targets are the inputs shifted by one. Here's the fix for my bug, we skip conversations that are too long. Here's best fit, the longest one that fits. The padding gets a mask of zero. Then inputs and targets, exactly like in pretraining. And wherever the mask is zero, we set the target to minus one. And minus one is the ignore index of the cross entropy loss in our model. So those positions simply don't count.

With packing, the padding goes down to about 8 percent, and that even includes the BOS tokens at the start of every conversation. And 42 percent of the tokens are trained on, instead of 25. That's almost twice as much signal from the same compute. Let's do one more sanity check, on our tiny France conversation, and look at the inputs and targets side by side.

In [26]:
small = sft_data_generator([conversation], batch_size=1, seq_len=12, buffer_size=1)
x, y, _ = next(small)
print([tokenizer.decode([t]) for t in x[0].tolist()])
print(y[0].tolist())

['<|endoftext|>', '<|user_start|>', 'What', ' is', ' the', ' capital', ' of', ' France', '?', '<|user_end|>', '<|assistant_start|>', 'Paris']
[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 40313, 50260]


So the inputs are the conversation without its last token. And the targets are minus one everywhere, except at the last two positions. After assistant start, the target is Paris, and after Paris, the target is assistant end. Those are the only two predictions that this conversation trains.

> **Exercise:** Change the dataloader so that a conversation that's too long gets cropped in a smarter way, keeping the last assistant answer, instead of being skipped. Is it worth it? Measure how many trained tokens you gain.

## The SFT training loop

Okay, and now the punchline of this whole stage. SFT is literally just pretraining with a different dataloader. Same model, same cross entropy loss, same AdamW optimizer, same training loop. The only differences are the minus ones in the targets, and a gentler learning rate. The model already knows a lot, we're just nudging its statistics, so we use a smaller learning rate than in pretraining, keep it constant for the first half, and then decay it to zero.

In [27]:
def get_lr(progress, lr=1e-4, warmdown_ratio=0.5):
    if progress < 1 - warmdown_ratio:
        return lr
    return lr * (1 - progress) / warmdown_ratio

print([f"{get_lr(p):.1e}" for p in [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]])

['1.0e-04', '1.0e-04', '1.0e-04', '5.0e-05', '2.0e-05', '0.0e+00']


Constant for the first half, then down to zero at the end. That's the schedule nanochat uses. And now, let's actually train, right here in the notebook. A fresh copy of GPT-2, AdamW, our dataloader with small rows of 256 tokens, and the training loop you've written ten times by now.

In [28]:
torch.manual_seed(1337)
student = GPT.from_pretrained()
optimizer = student.configure_optimizer(lr=1e-4)
loader = sft_data_generator(mixture, batch_size=4, seq_len=256)
for step in range(8):
    x, y, _ = next(loader)
    loss = student(x, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    print(f"step {step} | loss {loss.item():.4f} | trained tokens {(y >= 0).sum().item()}")
del student, optimizer

step 0 | loss 3.3616 | trained tokens 243


step 1 | loss 3.2339 | trained tokens 578


step 2 | loss 2.6976 | trained tokens 421


step 3 | loss 2.6803 | trained tokens 478


step 4 | loss 2.5765 | trained tokens 619


step 5 | loss 1.9085 | trained tokens 550


step 6 | loss 2.0697 | trained tokens 693


step 7 | loss 1.6392 | trained tokens 662


And the loss is going down. From 3.4 to 1.6 in eight steps. And notice how the number of trained tokens jumps around from batch to batch, that's because every batch has a different mix of conversations, so the loss of a single step is noisy.

So that's SFT. Everything else is engineering. I've put the dataloader, the schedule and the loop into chat_sft.py, plus command line arguments, a validation loss every hundred steps, gradient accumulation, bfloat16 and torch compile on a GPU, and saving the checkpoint at the end. In nanochat, you do one full epoch over the whole mixture, which on a single A100 is something like half an hour. But to show you that you don't need a GPU, I ran a smaller version on a plain four core CPU. 600 steps of four rows of 512 tokens, about an hour and a half.

In [ ]:
!python chat_sft.py --device-batch-size=4 --seq-len=512 --grad-accum=1 --num-iterations=600 --smoltalk=20000 --mmlu=20000

Output of the lecture's run:
```
Training on 74,946 conversations
step     0 | val loss 2.7305
step     1 | progress 0.0% | loss 11.8539 | lr 1.00e-04 | trained tokens 8 | dt 11.71s
step     2 | progress 0.2% | loss 2.6189 | lr 1.00e-04 | trained tokens 428 | dt 10.86s
step     3 | progress 0.3% | loss 3.1415 | lr 1.00e-04 | trained tokens 378 | dt 11.22s
step     4 | progress 0.5% | loss 8.2122 | lr 1.00e-04 | trained tokens 12 | dt 10.30s
step     5 | progress 0.7% | loss 3.7664 | lr 1.00e-04 | trained tokens 380 | dt 10.38s
...
step   100 | val loss 1.6287
step   200 | val loss 1.4900
step   300 | val loss 1.4400
step   400 | val loss 1.4042
step   500 | val loss 1.3579
...
step   600 | progress 99.8% | loss 0.7444 | lr 3.33e-07 | trained tokens 750 | dt 7.81s
step   600 | val loss 1.3377
Saved the SFT model to /root/.cache/nanochat-posttrain/chatsft.pt
```

Now look at the very first step. A loss of 11.9. Why so much higher than in our notebook run? Look at the number of trained tokens. Eight. In the whole first batch! Can you figure out why? It's best fit. At the start, the buffer is full, and the longest conversations in it are those long MMLU reading passages. So every row is one of them, and each one has exactly two green tokens, the letter and assistant end. And assistant end is one of our special tokens, the ones that start out with a loss of 15. Then in step two there are 428 trained tokens, and the loss is a normal 2.6. So the loss of a single step can be really misleading, and that's why we look at the validation loss, which goes down smoothly, from 2.73 to 1.34.

Okay, now the fun part. Let's talk to it. Here's a little chat function: it puts the question in a conversation, renders it for completion, and samples until assistant end.

In [29]:
from gpt import load_model
device = "cpu"
sft_model = load_model("sft", device).eval()
sft_engine = Engine(sft_model, tokenizer)

def chat(question, engine=sft_engine, temperature=0.0, seed=42, max_tokens=100):
    conversation = {"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": ""}]}
    ids = tokenizer.render_for_completion(conversation)
    out, _ = engine.generate_batch(ids, max_tokens=max_tokens, temperature=temperature, top_k=50, seed=seed)
    return tokenizer.decode(out[0][len(ids):])

print(chat("Hi! Who are you?", temperature=0.7, seed=1))

Hello! How can I help you today?


"Hello! How can I help you today?" And then it stopped. All by itself, it emitted assistant end. Compare that to the base model, which kept going forever. So the format, the turn taking, that's learned. Let's ask about France.

In [30]:
print(chat("What is the capital of France?"))

The capital of France is Paris, France. Paris is the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France


It answers directly now, as an assistant. But then it gets stuck in a loop. This is a classic failure of greedy decoding with small models: once it has said "the capital of France" twice, the most likely thing is to say it again. Sampling with a temperature helps, so let's try that.

In [31]:
print(chat("What is the capital of France?", temperature=0.7, seed=2, max_tokens=150))

We are asked to find the capital of France. Let me try a manual approach first.

First spell the word out:
France:f,a,l,e,r,n,e,s,t

Then count the occurrences of 'f':
1:f hit! count=1
2:l
3:e
4:r
5:n
6:e
7:s
8:t

This gives us 1.

Let me double check this using Python:

<|python_start|>'f'.count('f')<|python_end|><|output_start|>1<|output_end|>

Python gives us 1.

My final answer is:

#### 1


Ha! Look at this. "We are asked to find the capital of France. Let me try a manual approach first." And then it spells France, badly, counts the letter f, and calls python to double check. It's running the SpellingBee procedure on a geography question! Why? Remember the mixture. SpellingBee is more than a quarter of everything this model saw. And a lot of SpellingBee questions are short questions, like "what is the count of e in some word". So "what is the capital of France" looks like one of those on the surface, and the model pattern matches on the surface. If I sample answers to thirty real questions from the SmolTalk test set, only two start like this, so it's mostly short questions. But it's a great lesson. Every dataset in the mixture shapes the model, in ways that leak out beyond that dataset. When Karpathy added SpellingBee to nanochat, he noted in the commit that it probably uses too many of these examples. And on a GPU, with all of SmolTalk in the mixture, this mostly goes away.

In [32]:
print(chat("Why is the sky blue?", temperature=0.7, seed=1))

We are asked to find the cause of the sky blue phenomenon. According to meteorology, there are two types of meteors: the meteoroid and the meteorite. The meteoroid is a cloud of dust that collects dust and creates a blue glow. The meteorite is a cloud of molten metal that absorbs sunlight and returns it to the Earth. The cosmic rays from the cosmic ray shower are then released and create a blue glow. The blue glow is caused by the absorption of light and radiation from


And this one is kind of hilarious, meteoroids and molten metal. So let's be honest about what we have. This is a 124 million parameter model from 2019, finetuned for an hour and a half on a CPU. It doesn't know why the sky is blue, and it's happy to make things up with total confidence. The nanochat README says talking to its speedrun model is like talking to a kindergartener, and we're well below that. But the format is there, the turn taking is there, and as we'll see, on the narrow tasks we specifically trained for, it can do quite well.

> **Exercise:** Re-run SFT with five thousand SpellingBee examples instead of twenty thousand, and measure how often the model starts an ordinary answer with "We are asked to find". What happens to its SpellingBee accuracy?

## The inference engine

Okay, so far we've been using the Engine to sample, as a black box. Now let's write it. Sampling has two problems that we need to solve: it needs to be fast, because in reinforcement learning we're going to sample a lot, and it needs to run the python tool. Let's start with speed. Here's the simplest sampling loop you could write, the one from the GPT-2 video.

In [33]:
import time
@torch.no_grad()
def generate_naive(model, prompt, max_tokens):
    ids = torch.tensor([prompt])
    for _ in range(max_tokens):
        logits = model(ids)[:, -1, :] # run the WHOLE sequence through the model, every time
        ids = torch.cat([ids, logits.argmax(dim=-1, keepdim=True)], dim=1)
    return ids[0].tolist()

prompt = [tokenizer.bos] + tokenizer.encode("The meaning of life is")
t0 = time.time()
naive = generate_naive(model, prompt, 100)
print(f"naive: {time.time() - t0:.2f}s")

naive: 7.83s


Several seconds for 100 tokens. Why is it slow? Think about what this loop does. To get token number 7, it runs tokens 1 through 6 through the model. Then to get token 8, it runs tokens 1 through 7 through the model, all of them, again. But nothing about tokens 1 through 6 has changed! Because of the causal mask, a token can only look at tokens before it, so its keys and values in every layer are exactly the same as last time. We're recomputing them over and over, and the total work grows like the square of the length.

So the fix is to remember them. That's the KV cache. For every layer, we keep the keys and values of all the tokens so far, and when a new token comes in, we only compute the query, key and value for that one token, append its key and value to the cache, and let its query attend to everything in the cache. Now, this needs three small changes inside our GPT, and you should make them in your own gpt.py from the GPT-2 video. Change one is the cache itself. It's a tiny class that holds the keys and values, and update appends the new ones and returns everything. Change two is in the attention. If there's a cache, we append and get back all the keys and values. And here's a subtle one. If we feed in a single new token, we don't want a causal mask at all, because that one token is allowed to look at everything in the cache. And change three, the positions. The position embedding of the new token must start where the cache left off, otherwise every new token would think it's at position zero. And with that, here's sampling with a cache.

In [34]:
from gpt import KVCache

@torch.no_grad()
def generate_cached(model, prompt, max_tokens):
    kv_cache = [KVCache() for _ in range(model.config.n_layer)]
    logits = model(torch.tensor([prompt]), kv_cache=kv_cache)[:, -1, :] # prefill: the whole prompt, once
    out = list(prompt)
    for _ in range(max_tokens):
        token = logits.argmax(dim=-1).item()
        out.append(token)
        logits = model(torch.tensor([[token]]), kv_cache=kv_cache)[:, -1, :] # decode: one token at a time
    return out, kv_cache

t0 = time.time()
cached, kv_cache = generate_cached(model, prompt, 100)
print(f"cached: {time.time() - t0:.2f}s, same tokens: {cached[:len(naive)] == naive}")
print(kv_cache[0].k.shape)

cached: 2.20s, same tokens: True
torch.Size([1, 12, 106, 64])


First we run the whole prompt through once, which fills the cache. That's called the prefill. And then every step, we only feed in the one new token. Exactly the same tokens as the naive version, which is the first thing to always check, and about four times faster. And that gap grows with length. And here's the cache of the first layer: batch of one, twelve heads, 106 positions, which is our 6 prompt tokens plus 100 new ones, and 64 dimensions per head.

One more trick, which we'll need for reinforcement learning. Often we want many samples for the same prompt. So we do the prefill once, with a batch size of one, and then we just copy the cache for every sample, and decode all of them in parallel, as one batch.

Okay, now the second job, tools. Before I show you, think about it for a second. The model is sampling one token at a time. At some point it emits python start, then some tokens, then python end. What should happen, and what should the model see next?

Here's the idea. When we sample python start, we start recording the tokens. When we sample python end, we decode the recorded tokens into a string, and we run it. And then here's the trick. We put output start, the tokens of the result, and output end into a queue, and on the next steps, instead of using what the model sampled, we take tokens from the queue. We force them in. So the result appears in the model's own context, as if it had written it, and then it just continues from there. Let's write it.

In [35]:
from collections import deque
from engine import use_calculator

@torch.no_grad()
def generate_with_tools(model, prompt, max_tokens=256):
    s = tokenizer.encode_special
    kv_cache = [KVCache() for _ in range(model.config.n_layer)]
    logits = model(torch.tensor([prompt]), kv_cache=kv_cache)[:, -1, :]
    out, forced, expr = [], deque(), None
    for _ in range(max_tokens):
        token = forced.popleft() if forced else logits.argmax(dim=-1).item()
        out.append(token)
        if token == s("<|assistant_end|>"):
            break
        if token == s("<|python_start|>"):
            expr = [] # start recording the expression
        elif token == s("<|python_end|>") and expr is not None:
            result = use_calculator(tokenizer.decode(expr)) # run the tool...
            if result is not None: # ...and force its output into the stream
                forced.extend([s("<|output_start|>"), *tokenizer.encode(str(result)), s("<|output_end|>")])
            expr = None
        elif expr is not None:
            expr.append(token)
        logits = model(torch.tensor([[token]]), kv_cache=kv_cache)[:, -1, :]
    return out

question = {"messages": [{"role": "user", "content": "How many u are in the word eurytherm"}, {"role": "assistant", "content": ""}]}
print(tokenizer.decode(generate_with_tools(sft_model, tokenizer.render_for_completion(question))))

We are asked to find the number 'u' in the word 'eurytherm'. Let me try a manual approach first.

First spell the word out:
eurytherm:e,y,u,r,e,r,m,e,r

Then count the occurrences of 'u':
1:e
2:y
3:u hit! count=1
4:r
5:e
6:r
7:m
8:e
9:r

This gives us 1.

Let me double check this using Python:

<|python_start|>'eurytherm'.count('u')<|python_end|><|output_start|>1<|output_end|>

Python gives us 1.

My final answer is:

#### 1<|assistant_end|>


So every step, if there are forced tokens waiting, we take one, otherwise we use the model's choice. On python start we start recording, on python end we run it, and we queue up the result. And let's try it on our SFT model, on a spelling question it has never seen. So look at what happens. It spells the word, and it gets the spelling wrong! e, y, u, r. It roughly knows which letters are in there, but not in which order, because remember, it never sees the letters. Then it counts the u's in its own misspelled version and gets one, which happens to be right. Then it writes the python call, and in the python call it copies the word perfectly, because copying tokens is easy. And our code runs it and forces in the result. And the final answer is one. So the tool is robust to the model's own sloppiness.

Now, a word about safety. We're running code that a neural network wrote, so the calculator is deliberately very restricted. It allows numbers and arithmetic, without powers, or the count method of a string, and it evaluates without any builtins.

In [36]:
print(use_calculator("2*12"), use_calculator("'strawberry'.count('r')"))
print(use_calculator("__import__('os').system('ls')"), use_calculator("9**9**9"))

24 3
None None


So arithmetic and counting work, and an attempt to run a shell command, or to compute a gigantic number, just returns None. Never give a language model an unrestricted eval on your machine.

And the Engine in the repo is exactly what we just wrote, but for a whole batch of samples at once, with every row keeping its own little state, the prefill once and copy trick, and one more thing: it returns a mask, one for tokens the model sampled, and zero for tokens that were forced in. Remember that mask, because we'll need it for reinforcement learning. Let's sample eight answers to the same question.

In [37]:
example = SpellingBee(size=10, split="test")[6]
ids = tokenizer.render_for_completion(example)
out, masks = sft_engine.generate_batch(ids, num_samples=8, max_tokens=256, temperature=1.0, top_k=50, seed=0)
print(example["messages"][0]["content"])
for o, m in zip(out, masks):
    completion = tokenizer.decode(o[len(ids):])
    print(f"answer {extract_answer(completion)} | {len(o) - len(ids)} tokens, {m[len(ids):].count(0)} forced in by the tool")

How many u are in the word eurytherm
answer 1 | 154 tokens, 3 forced in by the tool
answer 1 | 154 tokens, 3 forced in by the tool
answer 1 | 172 tokens, 3 forced in by the tool
answer 1 | 154 tokens, 3 forced in by the tool
answer 1 | 174 tokens, 3 forced in by the tool
answer 1 | 148 tokens, 3 forced in by the tool
answer 1 | 154 tokens, 3 forced in by the tool
answer 1 | 166 tokens, 3 forced in by the tool


Every sample has three forced tokens, output start, the result, and output end, and they come out with different lengths, because we sampled with a temperature.

> **Exercise:** Add the three KV cache changes to your own GPT from the GPT-2 video, and check that your cached sampling gives exactly the same tokens as naive sampling. Then make the cache faster: instead of concatenating tensors every step, allocate the whole cache up front and write into it.

## Evaluation

Okay, so we have a model that seems to work. But "seems to work" is not good enough, we need numbers. And there are two kinds of evaluation in nanochat.

The first is multiple choice, like MMLU. And here's a neat thing: we don't even have to sample. We render the question, prime the assistant, and run one forward pass. Then we look at the logits at the last position, which is the model's prediction for the first token of its answer, but only at the logits of the allowed letters, A, B, C and D. Whichever is biggest, that's the answer. Let's write it, and compare the base model and our SFT model on 100 questions.

In [38]:
@torch.no_grad()
def multiple_choice_accuracy(model, task, n):
    correct = 0
    for i in range(n):
        conversation = task[i]
        ids = tokenizer.render_for_completion(conversation)
        logits = model(torch.tensor([ids]))[0, -1] # the logits for the assistant's first token
        letter_ids = [tokenizer.encode(letter)[0] for letter in conversation["letters"]]
        predicted = conversation["letters"][logits[letter_ids].argmax().item()]
        correct += predicted == conversation["messages"][-1]["content"]
    return correct / n

mmlu_test = MMLU("test")
print(f"base: {multiple_choice_accuracy(model, mmlu_test, 100):.2f}, sft: {multiple_choice_accuracy(sft_model, mmlu_test, 100):.2f}")

base: 0.29, sft: 0.26


One forward pass, the logits at the last position, restricted to the letters. And both are about 25 percent, which is exactly random guessing for four choices. And with only 100 questions, the noise on these numbers is about plus or minus five percent, so 29 versus 26 is not a real difference. Now this is important to understand. GPT-2 small simply doesn't know the answers to these questions. SFT can teach a format, but it can't teach knowledge that isn't there. For MMLU to move, you need a much stronger pretrained model, which is a big part of why nanochat trains its own base models.

The second kind is generative. We sample an answer, pull out the number after the hash signs, and compare. Let's do 20 SpellingBee questions the model has never seen.

In [39]:
bee_test = SpellingBee(size=20, split="test")
correct = 0
for i in range(20):
    conversation = bee_test[i]
    ids = tokenizer.render_for_completion(conversation)
    out, _ = sft_engine.generate_batch(ids, max_tokens=256, temperature=0.0)
    correct += bee_test.evaluate(conversation, tokenizer.decode(out[0][len(ids):]))
print(f"SpellingBee: {correct}/20")

SpellingBee: 9/20


Nine out of twenty. So about half. That's a narrow, procedural task, and SFT taught the model a procedure that it can actually execute, most of the time. In the repo, chat_eval.py has both of these, plus something nanochat calls ChatCORE: every accuracy is rescaled so that zero means random guessing and one means perfect, and then averaged into one number. Here's what I got on 100 problems of every task.

In [ ]:
!python chat_eval.py -i base -x 100
!python chat_eval.py -i sft -x 100

Output of the lecture's run:
```
$ python chat_eval.py -i base -x 100
ARC-Easy: 29.00%
ARC-Challenge: 26.00%
MMLU: 29.00%
GSM8K: 0.00%
SpellingBee: 0.00%
ChatCORE: 0.0240
$ python chat_eval.py -i sft -x 100
ARC-Easy: 24.00%
ARC-Challenge: 26.00%
MMLU: 26.00%
GSM8K: 0.00%
SpellingBee: 53.00%
ChatCORE: 0.1087
```

Multiple choice is at chance for both, as we saw. GSM8K is zero for both: the SFT model learned what a math solution looks like, with tool calls and hash signs, but it can't do the reasoning. And SpellingBee goes from zero to 53 percent. So the ChatCORE goes from 0.02 to 0.11, and almost all of that is SpellingBee. And 53 percent is not great, but it's not zero either, and that's exactly where we want to be for the next stage.

> **Exercise:** Evaluate the base model on MMLU with a few-shot prompt, using plain text instead of our chat format. How close can you get to the SFT model just by prompting?

## Reinforcement learning

Okay, and now the last stage, and I think the most exciting one. Why do we need it? Karpathy has a great analogy. Think of a textbook. Most of it is exposition, background knowledge. That's pretraining. Then there are worked examples, where an expert solves a problem step by step, and you imitate them. That's SFT. And at the end of every chapter, there are practice problems, with the final answer in the back of the book but no solution. You try, you fail, you try again, and you figure out what works for you. That's reinforcement learning.

And here's the deep reason why this matters. With SFT, we make the model imitate a solution that some person, or some other model, wrote. But that author has a very different brain. A step that's trivial for them might be a leap that our little model can't make in one token. And a step they wrote out carefully might be obvious to our model. We don't actually know the best sequence of tokens for this particular model. So the idea of RL is: don't tell it. Let it try many times, and whatever works, do more of that.

So let's make that precise. We have a question, the model samples an answer y, and we check it with our grader and get a reward R, one if it's correct and zero if not. And we want to maximize the expected reward, the average reward of the answers the model would sample. Now how do we get a gradient of an average over samples? Here's the trick, and it's worth deriving once. The expected reward is a sum over all possible answers of the probability of the answer times its reward, so its gradient is the sum of the reward times the gradient of the probability. Now the gradient of p is the same thing as p times the gradient of log p, that's just the chain rule backwards. And plugging that in, we get an expectation again! The average, over sampled answers, of the reward times the gradient of the log probability. And an expectation we can estimate by sampling. And the log probability of an answer is the sum of the log probabilities of its tokens, which is just minus the loss our model already computes.

So in code, it's almost embarrassingly simple. Sample some answers, and for each one, take the log probability of its tokens, multiply by its reward, and do gradient ascent on that. If you squint, it's just SFT on your own samples, weighted by how good they were. This is called the policy gradient, or REINFORCE.

But there's one problem. If rewards are zero or one, the wrong answers get a weight of zero, so they're simply ignored. And on easy questions, where every answer is right, everything gets pushed up, even though there's nothing to learn. So what we do is subtract a baseline. For every question, we sample a group of answers, say eight of them, and the advantage of each answer is its reward minus the mean reward of the group. So an answer that's better than average for this question has a positive advantage, and its probability gets pushed up. An answer that's worse than average gets pushed down. And you can show that subtracting a baseline doesn't change the expected gradient at all, it just makes it a lot less noisy. This is the group in GRPO, group relative policy optimization.

Before we do this on a transformer, let's build intuition on the tiniest possible example. Imagine a language model that produces a single token, out of four options, and only option two is correct. The whole model is four logits. We sample a group of sixteen attempts, reward the correct ones, compute advantages, and take a gradient step.

In [40]:
def reinforce_toy(logits, steps=40):
    torch.manual_seed(0)
    logits = torch.tensor(logits, requires_grad=True)
    optimizer = torch.optim.SGD([logits], lr=1.0)
    for step in range(steps + 1):
        probs = logits.softmax(dim=0)
        samples = torch.multinomial(probs, num_samples=16, replacement=True) # a group of 16 attempts
        rewards = (samples == 2).float()                                     # only option 2 is correct
        advantages = rewards - rewards.mean()
        loss = -(probs.log()[samples] * advantages).mean()                   # the policy gradient loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 10 == 0:
            print(f"step {step:2d} | p(correct) {probs[2].item():.4f} | mean reward {rewards.mean().item():.3f}")

reinforce_toy([0.0, 0.0, 0.0, 0.0])

step  0 | p(correct) 0.2500 | mean reward 0.250
step 10 | p(correct) 0.8508 | mean reward 0.812
step 20 | p(correct) 0.9490 | mean reward 0.938
step 30 | p(correct) 0.9673 | mean reward 0.688
step 40 | p(correct) 0.9801 | mean reward 0.938


The probability of the correct answer starts at a quarter, because all logits are zero, after ten steps it's at 85 percent, and it keeps climbing towards one. Nobody ever told it the answer. It just tried things and did more of what worked. Now what if the model starts out really bad, and the correct answer is very unlikely?

In [41]:
reinforce_toy([0.0, 0.0, -10.0, 0.0])

step  0 | p(correct) 0.0000 | mean reward 0.000
step 10 | p(correct) 0.0000 | mean reward 0.000
step 20 | p(correct) 0.0000 | mean reward 0.000
step 30 | p(correct) 0.0000 | mean reward 0.000
step 40 | p(correct) 0.0000 | mean reward 0.000


Nothing happens. The correct answer is never sampled, so the reward is always zero, so every advantage is zero, and there's no gradient. And this is the most important lesson about RL: it can only amplify things the model already does sometimes. It can't create a behavior from nothing. That's why we do SFT first, to get into a regime where the model sometimes succeeds. And the same is true at the other end: if all eight answers are right, every advantage is zero too. RL only learns from questions at the edge of the model's ability.

Okay, now the real thing, on our SFT model. Step one, rollouts. We take a SpellingBee question, render it for completion, and use our engine to sample a group of eight answers, at temperature one, because we want variety. Then we grade each one.

In [42]:
train_bee = SpellingBee(size=1000, split="train")
conversation = train_bee[6]
prompt = tokenizer.render_for_completion(conversation)
sequences, masks = sft_engine.generate_batch(prompt, num_samples=8, max_tokens=256, temperature=1.0, top_k=50, seed=0)
rewards = [train_bee.reward(conversation, tokenizer.decode(seq[len(prompt):])) for seq in sequences]
print(conversation["messages"][0]["content"])
print(rewards)

What is the count of 'o' in spook?
[1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0]


How many o's in spook. Four right, four wrong. That's a question right at the edge, perfect for learning. Now step two, the policy gradient. We pad the samples to the same length, and build inputs and targets exactly like in SFT. And here's where the mask from the engine comes in: we set the targets to minus one for the prompt, for the padding, and for the tokens that the tool forced in. We only train on the tokens that the model actually chose. Then advantages, the log probabilities, and the loss.

In [43]:
max_length = max(len(seq) for seq in sequences)
pad = tokenizer.encode_special("<|assistant_end|>")
ids = torch.tensor([seq + [pad] * (max_length - len(seq)) for seq in sequences])
mask = torch.tensor([m + [0] * (max_length - len(m)) for m in masks])
inputs, targets = ids[:, :-1], ids[:, 1:].clone()
targets[mask[:, 1:] == 0] = -1 # not the prompt, not the tool outputs, not the padding
rewards = torch.tensor(rewards)
advantages = rewards - rewards.mean()
print(advantages)
logp = -sft_model(inputs, targets, loss_reduction='none').view_as(inputs) # (8, T), 0 where target is -1
print((logp.sum(dim=1) / (targets >= 0).sum(dim=1)).tolist())
loss = -(logp * advantages.unsqueeze(-1)).sum() / (targets >= 0).sum()
loss.backward()
print(loss.item())
sft_model.zero_grad(set_to_none=True)

tensor([ 0.5000, -0.5000,  0.5000,  0.5000, -0.5000,  0.5000, -0.5000, -0.5000])


[-0.23993052542209625, -0.05462498217821121, -0.05465749278664589, -0.03908867761492729, -0.600749135017395, -0.039831407368183136, -0.1808803677558899, -0.06874782592058182]


-0.05557459592819214


There's the mask. The advantages are plus or minus a half, exactly like on the whiteboard. The model gives us the loss of every token, the negative log probability, so we negate it, and the ignored tokens come out as exactly zero. And here's the average log probability of each answer. The fifth answer, a wrong one, has a much lower log probability per token than the others, it's a bit of an unusual sample for this model. Then the loss: every token's log probability times the advantage of its answer, summed, divided by the number of trained tokens, and with a minus sign, because PyTorch minimizes and we want to maximize. And then backward, and an optimizer step. That's the whole algorithm.

Now if you've read about GRPO or PPO, you may wonder where everything else is. And nanochat makes four deliberate simplifications, written right at the top of the file. Number one, there's no KL penalty to the SFT model. That's a regularizer that keeps the model from drifting too far, and for a short run like ours it's not needed. Number two, no PPO ratio and clipping. Those exist because PPO takes several gradient steps on the same batch of samples, so the samples go stale and you have to correct for that. We take exactly one step per batch, so we're always on-policy, and the ratio would always be exactly one. Number three, we divide by the total number of tokens, not per answer, as the DAPO paper suggests, so long answers aren't down-weighted. And number four, we don't divide the advantage by the standard deviation. Put it all together, and what's left is good old REINFORCE, with a group mean baseline.

And chat_rl.py is exactly the two cells we just wrote, the rollouts, and the loss, plus an evaluation on held out questions every few steps. On a GPU you'd use sixteen questions per step and sixteen samples each. On my CPU, I did 40 steps of eight questions with eight samples each, which took about three hours, most of it sampling.

In [ ]:
!python chat_rl.py --task=SpellingBee --num-steps=40 --examples-per-step=8 --num-samples=8 --eval-examples=48 --lr=3e-5

Output of the lecture's run:
```
step    0 | eval pass@1 0.4427 | pass@8 0.9167
step    0 | reward 0.6406 | sampled tokens per answer 160.5 | lr 3.00e-05 | dt 193.4s
step    1 | reward 0.8750 | sampled tokens per answer 193.8 | lr 2.92e-05 | dt 349.8s
step    2 | reward 0.7812 | sampled tokens per answer 202.0 | lr 2.85e-05 | dt 248.2s
step    3 | reward 0.7344 | sampled tokens per answer 209.7 | lr 2.78e-05 | dt 252.6s
step    4 | reward 0.7812 | sampled tokens per answer 214.7 | lr 2.70e-05 | dt 252.2s
...
step   10 | eval pass@1 0.8385 | pass@8 1.0000
...
step   20 | eval pass@1 0.8672 | pass@8 0.9792
...
step   26 | reward 0.3438 | sampled tokens per answer 160.5 | lr 1.05e-05 | dt 153.6s
step   27 | reward 0.9844 | sampled tokens per answer 154.9 | lr 9.75e-06 | dt 135.3s
...
step   30 | eval pass@1 0.8776 | pass@8 1.0000
...
step   39 | reward 0.7500 | sampled tokens per answer 158.6 | lr 7.50e-07 | dt 142.3s
step   40 | eval pass@1 0.8750 | pass@8 1.0000
Saved the RL model to /root/.cache/nanochat-posttrain/chatrl.pt
```

So let's read this. Pass at one is the fraction of single samples that are correct, and before RL, it's 44 percent. Pass at eight is the fraction of questions where at least one of eight samples is correct, and it's 92 percent. So the model can already solve almost every question, it's just unreliable. And after only ten steps of RL, pass at one jumps to 84 percent, and ends at 87 and a half. This is a really common pattern. RL takes what the model can do some of the time, and makes it do it most of the time. It turns pass at k into pass at one.

And the training reward is really noisy, look at step 26. That's because every step only has eight questions, and some steps just get harder words. Always trust a held out evaluation, not the training reward. And one more fun thing. The answers get shorter, from about 210 tokens to about 155. Nobody asked for that. But many long answers were ones where the model got lost, misspelling a word into a long string of garbage letters, and those got zero reward, so RL pushed them down. And with greedy decoding on 100 test questions, SpellingBee goes from 53 percent after SFT to 86 percent after RL.

So now, finally, the famous question. Eight samples from the SFT model and from the RL model, and their final answers.

In [44]:
from tasks import extract_answer
rl_model = load_model("rl", device).eval()
rl_engine = Engine(rl_model, tokenizer)

def answers(question, engine):
    ids = tokenizer.render_for_completion({"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": ""}]})
    out, _ = engine.generate_batch(ids, num_samples=8, max_tokens=256, temperature=1.0, top_k=50)
    return [extract_answer(tokenizer.decode(o[len(ids):])) for o in out]

for question in ["How many r are in the word 'strawberry'?", "How many e are in cheese?", "How many r are in strawberry?"]:
    for name, engine_ in [("sft", sft_engine), ("rl", rl_engine)]:
        print(f"{name:3s} | {question:42s} | {answers(question, engine_)}")

sft | How many r are in the word 'strawberry'?   | ['1', '3', '2', '2', '2', '2', '2', '1']


rl  | How many r are in the word 'strawberry'?   | ['3', '3', '3', '3', '3', '3', '3', '3']


sft | How many e are in cheese?                  | ['2', '5', '3', '2', '4', '3', '3', '3']


rl  | How many e are in cheese?                  | ['3', '3', '3', '3', '3', '3', '3', '3']


sft | How many r are in strawberry?              | ['2', '1', '1', '2', '2', '1', '2', '3']


rl  | How many r are in strawberry?              | ['4', '2', '3', '3', '1', '2', '2', '1']


Before RL, three r's in strawberry only once out of eight. After RL, eight out of eight. Same for cheese. That's RL. But look at the last question, without the quotes and without "the word". Both models fall apart. What's going on? Let's look.

In [45]:
print(chat("How many r are in strawberry?", engine=rl_engine, max_tokens=40))
print([tokenizer.decode([t]) for t in tokenizer.encode("How many r are in strawberry?")])
print([tokenizer.decode([t]) for t in tokenizer.encode("How many r are in the word 'strawberry'?")])

We are asked to find the number 'r' in the word 'sweet'. Let me try a manual approach first.

First spell the word out:
sweet:s,t
['How', ' many', ' r', ' are', ' in', ' strawberry', '?']
['How', ' many', ' r', ' are', ' in', ' the', ' word', " '", 'st', 'raw', 'berry', "'?"]


It says: we are asked to find the number of r's in the word "sweet". Sweet! It didn't even copy the right word. And the tokens tell you why. Without quotes, the word is "space strawberry", one single token. With quotes, it's "st", "raw", "berry", exactly the tokens the model has to write in its spelling and in the python call. So with quotes, it just has to copy three tokens. Without quotes, it has to know how one single token is spelled, and write it as a different set of tokens, and it doesn't know that, so it reaches for something nearby in meaning: sweet. Our SpellingBee data mostly has rare words, which are several tokens either way, so the model never learned to unpack a common single-token word. The tokenizer, once again. And the fix is what you'd do in practice: now that we understand the failure, add training data for exactly this case. You look at the failures, you understand them, and you go back and fix the data.

> **Exercise:** Write the RL training loop yourself, using the two cells above: a loop over questions, an AdamW optimizer with a small learning rate, and an evaluation of pass@1 every ten steps. Then try changing the reward, for example a small penalty for long answers, and see what the model does. Careful: RL will find any loophole in your reward.

## Scaling up, and what's next

Okay, so let's zoom out. We took a base model, a document simulator, and we gave it a format for conversations, with special tokens and a loss mask. We built a mixture of data where every part teaches something specific, a packing dataloader, and we ran SFT, which is just pretraining on that data. We wrote an inference engine with a KV cache and tool use, evaluations, and reinforcement learning with a group baseline. And you've now written every one of these pieces yourself.

And this maps one to one onto nanochat. Our tokenizer, tasks, engine, and the chat_sft, chat_rl, chat_eval and chat_cli scripts all have the same names there. The real thing pretrains its own base model, whose size you pick with one depth parameter, it uses the Muon optimizer, bfloat16, eight GPUs with data parallel, and in nanochat, RL is done on GSM8K. But the ideas are the ones you just built. If you have a GPU node, it's one script.

```bash
git clone https://github.com/karpathy/nanochat && cd nanochat
bash runs/speedrun.sh                                        # tokenizer, pretraining, SFT, eval (~2 hours on 8XH100)
torchrun --standalone --nproc_per_node=8 -m scripts.chat_rl  # reinforcement learning on GSM8K
python -m scripts.chat_cli -i rl                             # talk to it
```

And there's a lot we didn't cover. We only did RL where we can check the answer automatically. But what about a poem, or a joke? There's no answer key. That's where RLHF comes in, where you train a separate reward model on human preferences and do RL against it. Or DPO, which skips the reward model. But a reward model is itself a neural net, and if you run RL against it for long, the policy finds adversarial examples that fool it. And there's the whole world of reasoning models, where you run RL for a long time on hard math and code, and the model discovers on its own that it should think for a long time, try different approaches and check its work. Our little SpellingBee procedure is a tiny cartoon of that.

Okay, so that's it for today. Take the code, run it, break it, add your own tasks, and see what your model learns. See you next time.